In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import sys
import logging
import os
import random
import time
import re
import math
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from bs4 import BeautifulSoup
from dateutil import parser as date_parser

# --- Selenium ---
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from webdriver_manager.chrome import ChromeDriverManager

# --- Requests ---
import requests
import warnings
from urllib3.exceptions import InsecureRequestWarning

warnings.simplefilter('ignore', InsecureRequestWarning)

# ===============================================================
#  全域設定
# ===============================================================

try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

# 依然寫入主資料庫，方便您測試完直接用
DATA_FILENAME = os.path.join(BASE_DIR, 'news_dataset_v5.csv')
LOG_FILENAME = os.path.join(BASE_DIR, 'scraper_setn_test.log')
V5_COLUMNS = ['url', 'media', 'category', 'publish_time', 'title', 'content', 'fetch_time', 'label']

# SETN 測試目標
LIMIT_L1 = 50
LIMIT_L0 = 75

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILENAME, encoding='utf-8', mode='a'),
        logging.StreamHandler(sys.stdout)
    ]
)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Referer': 'https://www.google.com/'
}

# ===============================================================
#  SETN 專用設定
# ===============================================================

SETN_CONFIG = {
    'name': 'SETN',
    'hot_url': 'https://www.setn.com/ViewAll.aspx?PageGroupID=6',
    'latest_url': 'https://www.setn.com/ViewAll.aspx',
    'base_url': 'https://www.setn.com',
    # 包含列表模式與圖文模式的選擇器
    'hot_selector': ['h3.view-li-title a', 'div.news-list-group a.gt', 'div.col-sm-12 h3 a'],
    'latest_selector': ['h3.view-li-title a', 'div.col-sm-12 > a']
}

CONTENT_SELECTORS = [
    'div.article-content', 'div.story', 'div.article-body', 
    'div[itemprop="articleBody"]', 'section.article-content__editor',
    'div.news_content', 'article', 'div.text', 'div.p-note',
    'div.fncnews-content', 'div.post-content'
]

# ===============================================================
#  工具模組
# ===============================================================

def clean_category(cat_str):
    if not cat_str or cat_str == "即時": return "即時"
    if '>' in cat_str: cat_str = cat_str.split('>')[-1]
    return cat_str.strip()

def format_time_strict(time_str):
    now = datetime.now()
    if not time_str or str(time_str).lower() in ['nan', 'n/a', 'none']:
        return now.strftime("%Y/%-m/%-d %-I:%M:%S %p")
    try:
        raw = str(time_str).replace("發布時間：", "").replace("更新時間：", "").strip()
        if "小時前" in raw:
            hours = int(re.search(r'(\d+)', raw).group(1))
            dt = now - timedelta(hours=hours)
        elif "分鐘前" in raw:
            mins = int(re.search(r'(\d+)', raw).group(1))
            dt = now - timedelta(minutes=mins)
        elif "剛剛" in raw:
            dt = now
        else:
            dt = date_parser.parse(raw, fuzzy=True)
        
        hour_12 = int(dt.strftime("%I"))
        time_part = f"{hour_12}:{dt.strftime('%M:%S %p')}"
        return f"{dt.year}/{dt.month}/{dt.day} {time_part}"
    except:
        return now.strftime("%Y/%-m/%-d %-I:%M:%S %p")

def parse_html_content(html_text):
    soup = BeautifulSoup(html_text, 'lxml')
    content = np.nan
    for sel in CONTENT_SELECTORS:
        elm = soup.select_one(sel)
        if elm:
            for tag in elm(['script', 'style', 'iframe', 'div.knn_related', 'div.fb-root', 'div.author', 'div.ad']):
                tag.decompose()
            text = elm.get_text(separator=' ', strip=True)
            text = re.sub(r'\s+', ' ', text)
            if len(text) > 30:
                content = text
                break
    
    category = "即時"
    meta_sec = soup.select_one('meta[property="article:section"]')
    if meta_sec and meta_sec.get('content'):
        category = meta_sec.get('content')
    else:
        crumbs = soup.select('div.breadcrumb a, nav a, div.guide a')
        if len(crumbs) >= 2:
            txt = crumbs[1].get_text(strip=True)
            if len(txt) < 15: category = txt

    raw_time = None
    time_elm = soup.select_one('meta[property="article:published_time"], meta[name="pubdate"], time, span.time, div.date, div.meta-info time')
    if time_elm:
        raw_time = time_elm.get('content') if time_elm.has_attr('content') else time_elm.get_text(strip=True)
    
    return clean_category(category), format_time_strict(raw_time), content

class HumanNavigator:
    def __init__(self, driver):
        self.driver = driver
        self.action = ActionChains(driver)

    def random_sleep(self, min_s=1.5, max_s=3.0):
        time.sleep(random.uniform(min_s, max_s))

    def smooth_scroll(self):
        try:
            current_pos = self.driver.execute_script("return window.pageYOffset;")
            target_distance = random.randint(800, 1200)
            steps = random.randint(10, 20)
            for _ in range(steps):
                move = (target_distance / steps) + random.uniform(-5, 5) 
                current_pos += move
                self.driver.execute_script(f"window.scrollTo(0, {current_pos});")
                time.sleep(0.02)
        except: pass

class NewsSpider:
    def __init__(self):
        self.options = Options()
        self.options.add_argument("--headless")
        self.options.add_argument("--disable-gpu")
        self.options.add_argument("--no-sandbox")
        self.options.add_argument("--window-size=1920,1080")
        self.options.add_argument(f"user-agent={HEADERS['User-Agent']}")
        prefs = {"profile.managed_default_content_settings.images": 2}
        self.options.add_experimental_option("prefs", prefs)
        self.driver = None
        self.human = None
        self.existing_urls = self._get_existing_urls()

    def _get_existing_urls(self):
        if os.path.exists(DATA_FILENAME):
            try:
                df = pd.read_csv(DATA_FILENAME)
                if 'url' in df.columns: return set(df['url'].tolist())
            except: pass
        return set()

    def start(self):
        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service, options=self.options)
        self.driver.set_page_load_timeout(45)
        self.human = HumanNavigator(self.driver)

    def stop(self):
        if self.driver:
            try: self.driver.quit()
            except: pass
            self.driver = None

    def fetch_content_smart(self, url):
        # 1. Requests
        try:
            res = requests.get(url, headers=HEADERS, timeout=6, verify=False)
            if res.status_code == 200:
                cat, t, c = parse_html_content(res.text)
                if isinstance(c, str): return cat, t, c
        except: pass

        # 2. Selenium Fallback
        try:
            self.driver.execute_script("window.open('');")
            self.driver.switch_to.window(self.driver.window_handles[-1])
            self.driver.get(url)
            self.human.random_sleep(1.0, 2.0)
            html = self.driver.page_source
            self.driver.close()
            self.driver.switch_to.window(self.driver.window_handles[0])
            return parse_html_content(html)
        except:
            try:
                if len(self.driver.window_handles) > 1:
                    self.driver.close()
                    self.driver.switch_to.window(self.driver.window_handles[0])
            except: pass
        
        return "即時", format_time_strict(None), np.nan

    def crawl_task(self, site_config, label, limit):
        url = site_config['hot_url'] if label == 1 else site_config['latest_url']
        selectors = site_config['hot_selector'] if label == 1 else site_config['latest_selector']
        site_name = site_config['name']

        logging.info(f"      L{label} 任務啟動 (目標: {limit} 筆)...")
        try:
            self.driver.get(url)
            self.human.random_sleep(2, 4)

            collected = []
            seen_links = set()
            
            # 1. 捲動掃描
            scroll_target = int((limit / 8) * 1.5) + 3
            for _ in range(scroll_target):
                self.human.smooth_scroll()
                self.human.random_sleep(1.0, 1.5)
                soup = BeautifulSoup(self.driver.page_source, 'lxml')
                found_count = 0
                for sel in selectors:
                    found_count += len(soup.select(sel))
                if found_count >= limit * 1.2: break
            
            # 2. 解析
            soup = BeautifulSoup(self.driver.page_source, 'lxml')
            elements = []
            for sel in selectors:
                found = soup.select(sel)
                if found: elements.extend(found)

            unique_elements = []
            _temp_seen = set()
            for el in elements:
                h = el.get('href')
                if h and h not in _temp_seen:
                    _temp_seen.add(h)
                    unique_elements.append(el)
            
            logging.info(f"      -> 掃描到 {len(unique_elements)} 個連結，開始抓取...")

            # 3. 逐筆抓取
            for item in unique_elements:
                if len(collected) >= limit: break
                
                try:
                    link = item.get('href')
                    title = item.get('title') or item.get_text(strip=True)
                    if not link or not title or len(title) < 4: continue
                    
                    if link.startswith('/'):
                        base = site_config['base_url']
                        full_url = base.rstrip('/') + '/' + link.lstrip('/')
                    elif link.startswith('http'):
                        full_url = link
                    else: continue
                    
                    # *** SETN 專用修復：如果網址有 NewsID，不執行 split('?') ***
                    if 'setn.com' in full_url and 'NewsID=' in full_url:
                        clean_url = full_url
                    else:
                        clean_url = full_url.split('?')[0]

                    if clean_url in seen_links: continue
                    seen_links.add(clean_url)

                    # 去重檢查
                    is_exist = False
                    for exist in self.existing_urls:
                        if clean_url in str(exist):
                            is_exist = True; break
                    
                    if label == 0 and is_exist: continue 

                    # 抓內文
                    cat, t_str, content = self.fetch_content_smart(full_url)
                    
                    fetch_time = datetime.now().strftime("%Y/%-m/%-d %-I:%M:%S %p")
                    
                    collected.append({
                        'url': full_url,
                        'media': site_name,
                        'category': cat,
                        'publish_time': t_str,
                        'title': title,
                        'content': content,
                        'fetch_time': fetch_time,
                        'label': label
                    })
                    
                    status = "OK" if isinstance(content, str) else "NaN"
                    print(f"         [L{label}] {title[:8]}... ({status})")
                    
                    time.sleep(random.uniform(0.5, 1.0))

                except Exception: continue
            
            return collected

        except Exception as e:
            logging.error(f"      🔥 任務錯誤: {e}")
            return []

# ===============================================================
#  主程式 (SETN Only)
# ===============================================================

def main():
    logging.info(f"🚀 T-Forecast (SETN 專用測試版) 啟動")
    
    spider = NewsSpider()
    site_config = SETN_CONFIG
    site_name = site_config['name']
    
    logging.info(f"--- 處理媒體: {site_name} ---")
    
    spider.start()
    all_data = []
    
    try:
        # 2. 抓熱門 (L1)
        l1_data = spider.crawl_task(site_config, label=1, limit=LIMIT_L1)
        all_data.extend(l1_data)
        logging.info(f"      ✅ {site_name} L1 完成: {len(l1_data)} 筆")
        
        spider.human.random_sleep(2, 3)
        
        # 3. 抓即時 (L0)
        l0_data = spider.crawl_task(site_config, label=0, limit=LIMIT_L0)
        all_data.extend(l0_data)
        logging.info(f"      ✅ {site_name} L0 完成: {len(l0_data)} 筆")
        
    except Exception as e:
        logging.error(f"   🔥 {site_name} 發生錯誤: {e}")
    
    spider.stop()
    
    if all_data:
        df = pd.DataFrame(all_data)
        for col in V5_COLUMNS:
            if col not in df.columns: df[col] = np.nan
        df = df[V5_COLUMNS]
        try:
            df.to_csv(DATA_FILENAME, mode='a', header=False, index=False, encoding='utf-8-sig')
            logging.info(f"   💾 {site_name} 存檔成功 (共 {len(all_data)} 筆)")
        except:
            df.to_csv(f"backup_{site_name}_{int(time.time())}.csv", index=False)

    logging.info("🏁 SETN 任務結束")

if __name__ == "__main__":
    main()

2025-12-09 13:32:43,831 - INFO - 🚀 T-Forecast (SETN 專用測試版) 啟動
2025-12-09 13:32:43,858 - INFO - --- 處理媒體: SETN ---
2025-12-09 13:32:43,858 - INFO - ====== WebDriver manager ======
2025-12-09 13:32:44,127 - INFO - Get LATEST chromedriver version for google-chrome
2025-12-09 13:32:44,392 - INFO - Get LATEST chromedriver version for google-chrome
2025-12-09 13:32:44,578 - INFO - Driver [/Users/xiaobing/.wdm/drivers/chromedriver/mac64/143.0.7499.40/chromedriver-mac-arm64/chromedriver] found in cache
2025-12-09 13:32:45,456 - INFO -       L1 任務啟動 (目標: 50 筆)...
2025-12-09 13:32:54,581 - INFO -       -> 掃描到 42 個連結，開始抓取...
         [L1] 宏台若復交　學者... (OK)
         [L1] 藍白批行政院不執... (OK)
         [L1] 韓入境卡把台灣列... (OK)
         [L1] 藍強推助理費除罪... (OK)
         [L1] 青年憂AI發展影... (OK)
         [L1] 賴、柯關係回不去... (OK)
         [L1] 新／遭黑函爆「婚... (OK)
         [L1] 又封殺！藍白擋政... (OK)
         [L1] 中國不滿美國安戰... (OK)
         [L1] 「全民安全指引」... (OK)
         [L1] 助理費除罪藍營不... (OK)
         [L1] 賴清德用中文、日... (OK)
      